In [1]:
import ast
import os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
from tqdm import tqdm
import multiprocessing
import warnings
import time

warnings.filterwarnings('ignore')

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
all_data = []

for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    os.chdir(folder)
    replicon_data = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    replicon_data['genus'] = genus_name
    all_data.append(replicon_data)

all_data = pd.concat(all_data, ignore_index=True)

In [3]:
name_dict = {'accession': 'contig',
             'average plasmid fraction-original': 'plasmidness-original',
             'average plasmid fraction-pident_90': 'plasmidness-pident_90',
             'average plasmid fraction-pident_95': 'plasmidness-pident_95',
             'cp-label': 'original category',
            }
all_data.rename(columns=name_dict, inplace=True)
all_data = all_data[['contig', 'organism', 'genus', 'size', 'plasmidness-original',
       'plasmidness-pident_90', 'plasmidness-pident_95', 'original category', 'category-pident_90', 
       'GC_content', 'topology']]

target_dir = '/active-data/analysis_results/chr_pla/genus/suptables'
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)
all_data.to_csv('replicon_plasmidness_data.tsv', sep='\t', index=False)